In [1]:
# W2D4: Train/Test Split & Cross-Validation — leakage-safe and MLflow tracked.
# MLflow provides MLOps evidence. CrewAI/LangGraph can orchestrate downstream;
# Ragas applies only when evaluating a RAG application, not tabular classification.
from pathlib import Path
import json

import mlflow
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'week2_Notebooks' else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'w2d4_validation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{(OUTPUT_DIR / 'mlflow.db').as_posix()}")
mlflow.set_experiment('w2d4_train_test_cross_validation')

# CIA Full Stack Mentor review 1: stratify the holdout and verify no row overlap.
# Reproducible, imbalanced tabular data; stratification preserves the target ratio.
features, target = make_classification(n_samples=800, n_features=10, n_informative=6, n_redundant=2,
    weights=[0.7, 0.3], class_sep=1.1, flip_y=0.02, random_state=RANDOM_STATE)
X = pd.DataFrame(features, columns=[f'feature_{i}' for i in range(features.shape[1])])
y = pd.Series(target, name='target')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
assert set(X_train.index).isdisjoint(X_test.index), 'Train and test rows must not overlap.'
assert abs(y_train.mean() - y_test.mean()) < 0.02, 'Stratification should preserve class balance.'

# CIA Full Stack Mentor review 2: keep scaling inside CV and reserve test data for final scoring.
# Scaling stays in the pipeline, so each CV fold fits it only on its training rows.
pipeline = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_results = []
for fold, (train_idx, validation_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_fold_train, X_fold_valid = X_train.iloc[train_idx], X_train.iloc[validation_idx]
    y_fold_train, y_fold_valid = y_train.iloc[train_idx], y_train.iloc[validation_idx]
    pipeline.fit(X_fold_train, y_fold_train)
    prediction = pipeline.predict(X_fold_valid)
    probability = pipeline.predict_proba(X_fold_valid)[:, 1]
    fold_results.append({'fold': fold, 'accuracy': accuracy_score(y_fold_valid, prediction),
        'precision': precision_score(y_fold_valid, prediction, zero_division=0),
        'recall': recall_score(y_fold_valid, prediction, zero_division=0),
        'f1': f1_score(y_fold_valid, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_fold_valid, probability)})
fold_metrics = pd.DataFrame(fold_results)

# Fit once on all training data; score the untouched holdout exactly once.
pipeline.fit(X_train, y_train)
test_prediction = pipeline.predict(X_test)
test_probability = pipeline.predict_proba(X_test)[:, 1]
holdout_metrics = {'test_accuracy': accuracy_score(y_test, test_prediction),
    'test_precision': precision_score(y_test, test_prediction, zero_division=0),
    'test_recall': recall_score(y_test, test_prediction, zero_division=0),
    'test_f1': f1_score(y_test, test_prediction, zero_division=0),
    'test_roc_auc': roc_auc_score(y_test, test_probability)}
cv_metrics = {f'cv_{metric}_{stat}': float(fold_metrics[metric].agg(stat))
              for metric in fold_metrics.columns if metric != 'fold' for stat in ('mean', 'std')}
summary = {**cv_metrics, **holdout_metrics}

# MLOps evidence: MLflow parameters/metrics and exportable result artifacts.
fold_metrics.to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)
(OUTPUT_DIR / 'metrics_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
with mlflow.start_run(run_name='logistic_regression_stratified_cv'):
    mlflow.log_params({'model': 'LogisticRegression', 'split_strategy': 'stratified_holdout', 'test_size': 0.25,
        'cv_strategy': 'StratifiedKFold', 'cv_folds': 5, 'random_state': RANDOM_STATE,
        'train_rows': len(X_train), 'test_rows': len(X_test)})
    mlflow.log_metrics(summary)
    mlflow.log_artifact(str(OUTPUT_DIR / 'fold_metrics.csv'))
    mlflow.log_artifact(str(OUTPUT_DIR / 'metrics_summary.json'))

display(fold_metrics.round(3))
print(pd.Series(summary).round(3).to_string())
assert len(fold_metrics) == 5 and bool(fold_metrics.drop(columns='fold').apply(lambda scores: scores.between(0, 1).all()).all())
assert all(0 <= value <= 1 for value in holdout_metrics.values())
print('Self-review complete: stratified split, no overlap, leakage-safe CV pipeline, MLflow evidence, checks passed.')

2026/09/17 19:13:51 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\cynarisis-internship\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


,fold,accuracy,precision,recall,f1,roc_auc
0,1,0.817,0.778,0.568,0.656,0.880
1,2,0.867,0.784,0.784,0.784,0.902
2,3,0.858,0.812,0.703,0.754,0.892
3,4,0.833,0.793,0.622,0.697,0.877
4,5,0.850,0.806,0.676,0.735,0.901


cv_accuracy_mean     0.845
cv_accuracy_std      0.020
cv_precision_mean    0.795
cv_precision_std     0.015
cv_recall_mean       0.670
cv_recall_std        0.082
cv_f1_mean           0.725
cv_f1_std            0.050
cv_roc_auc_mean      0.890
cv_roc_auc_std       0.012
test_accuracy        0.830
test_precision       0.800
test_recall          0.590
test_f1              0.679
test_roc_auc         0.870
Self-review complete: stratified split, no overlap, leakage-safe CV pipeline, MLflow evidence, checks passed.
